# OpenJev with Ternary Bonsai 2 27B

Try **Ternary-Bonsai-2-27B-PTQ1_0.gguf** on a Colab GPU. The language weights are about **5.95 GB**. OpenJev scores the token `yes` for each candidate and generates zero answer tokens.

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Start with a fresh session and choose **Runtime → Run all**.
3. Change the message and rerun the model-loading and example cells to experiment.

This notebook uses Prism's required llama.cpp runtime and a short 2,048-token context. It needs no API key. The first run downloads the runtime and weights. This is a **text example**: OpenJev's GGUF backend does not yet load Bonsai's optional vision tower. For images, use the [Qwen notebook](https://colab.research.google.com/github/mjdileep/OpenJev/blob/main/notebooks/OpenJev_Quickstart.ipynb).

[Model card](https://huggingface.co/prism-ml/Ternary-Bonsai-2-27B-gguf) · [GitHub](https://github.com/mjdileep/OpenJev)


## 1. Install

The setup downloads checksum-verified Prism binaries and matching Python bindings. It keeps them in a separate runtime directory. Stock llama.cpp cannot load this model's `PTQ1_0` format. Run this cell once per fresh session; if you have already imported `llama_cpp`, restart the session first.


In [ ]:
import runpy
import urllib.request
from pathlib import Path

OPENJEV_REF = "37f2af0f7bc7a3ed05cb7da21c651ba97abfce0f"
OPENJEV_PACKAGE = "openjev @ git+https://github.com/mjdileep/OpenJev.git@" + OPENJEV_REF
DEPENDENCIES = (
    '"transformers==5.17.0" "huggingface-hub>=0.34,<2" '
    "numpy diskcache jinja2 typing_extensions"
)
%pip install -q "{OPENJEV_PACKAGE}" {DEPENDENCIES}

setup_path = Path("/content/openjev_bonsai_runtime.py")
setup_url = f"https://raw.githubusercontent.com/mjdileep/OpenJev/{OPENJEV_REF}/examples/bonsai_runtime.py"
urllib.request.urlretrieve(setup_url, setup_path)
prism_native = runpy.run_path(str(setup_path))["prepare"]()


## 2. Load the model

The defaults pin your requested GGUF and its matching tokenizer. The tokenizer comes from Prism's MLX repository; only tokenizer files are downloaded from that repository. Inference uses the GGUF weights on CUDA.

To change models, set `MODEL_ID`, `GGUF_FILE`, and `TOKENIZER_ID` to matching repositories/files, and update or clear the corresponding revisions. Keep the pinned runtime for Bonsai 2. Thinking is disabled for direct `yes`/`no` scoring.


In [ ]:
# @title Model settings
import gc
import json
import subprocess

from openjev import Choice, DecisionEngine, Noul, Score
from openjev.cli import benchmark, compare_results

MODEL_ID = "prism-ml/Ternary-Bonsai-2-27B-gguf"  # @param {type:"string"}
GGUF_FILE = "Ternary-Bonsai-2-27B-PTQ1_0.gguf"  # @param {type:"string"}
MODEL_REVISION = "6ed5e12bf84b7a63069882c91dd9e9218647d17b"  # @param {type:"string"}
TOKENIZER_ID = "prism-ml/Ternary-Bonsai-2-27B-mlx-2bit"  # @param {type:"string"}
TOKENIZER_REVISION = "3f926b415992eaa2ae9dd7b573706494d6bbf787"  # @param {type:"string"}
CONTEXT_TOKENS = 2048  # @param {type:"integer"}

previous_engine = globals().pop("engine", None)
if previous_engine is not None:
    previous_engine.close()
    del previous_engine
    gc.collect()

engine = DecisionEngine.from_pretrained(
    MODEL_ID,
    backend="gguf",
    filename=GGUF_FILE,
    revision=MODEL_REVISION or None,
    tokenizer=TOKENIZER_ID,
    tokenizer_revision=TOKENIZER_REVISION or None,
    device="cuda",
    n_gpu_layers=-1,
    n_ctx=CONTEXT_TOKENS,
    prefill_chunk_size=128,
    score_mode="full",
)
print("Ready:", MODEL_ID, "/", GGUF_FILE)
print("Verdict token IDs:", engine.backend.positive_id, engine.backend.negative_id)
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
    check=True,
)


## 3. Route a support message

Edit `MESSAGE` to try your own text. The questions match the Qwen quickstart.


In [ ]:
# @title Your message
MESSAGE = "Help! My payouts have been failing for three days."  # @param {type:"string"}

questions = {
    "department": Choice(
        "Which team should handle this?",
        {
            "billing": "Payments, invoicing, refunds",
            "technical": "Bugs, outages, integrations",
            "sales": "Pricing, upgrades, new accounts",
        },
    ),
    "urgent": Noul(
        "Does the message convey urgency?",
        {"true": "Explicitly time-sensitive", "false": "No urgency expressed"},
    ),
    "frustration": Score("How frustrated is the customer?", ["Calm", "Frustrated", "Very angry"]),
}
text_result = engine.decide(MESSAGE, questions)
answers = text_result.answers
print("Team:", answers["department"]["choice"])
print("Urgency support:", round(answers["urgent"]["noul"], 4))
frustration = answers["frustration"]
frustration_distribution = {
    frustration["legend"][int(level)]: weight
    for level, weight in frustration["probabilities"].items()
}
print("Frustration label:", max(frustration_distribution, key=frustration_distribution.get))
print("Frustration average (0=calm, 1=frustrated, 2=very angry):", round(frustration["score"], 3))
print("Frustration distribution:", json.dumps(frustration_distribution, indent=2))
print("Team distribution:", json.dumps(answers["department"]["probabilities"], indent=2))
print("Reused input tokens:", text_result.usage.reused_input_tokens)
print("Generated tokens:", text_result.usage.generated_tokens)

### What these numbers mean

The default raw support is the full-vocabulary next-token probability `P("yes")`. A choice distribution normalizes the independently scored candidates; it is **not a calibrated probability of correctness**. Low support for every candidate can still produce a confident-looking normalized choice. Inspect the raw evidence below before choosing a threshold for a real workflow.

`Score` returns a weighted average of the ordered levels. Here it is `0 × P(Calm) + 1 × P(Frustrated) + 2 × P(Very angry)`, using the normalized candidate scores. A value such as `1.053` is on the **0–2 rating scale**. The notebook prints the leading label and the whole distribution too: an average near 1 can also arise from a split between calm and very angry.

In [ ]:
print(json.dumps(text_result.answers["department"]["candidates"], indent=2))

## 4. Check caching

This compares identical questions with caching enabled and disabled, after warming both paths. It also checks whether reversing candidate order changes the scores. The benchmark excludes installation and model loading. Snapshot copies can outweigh computation savings on short prompts; inspect the measured result.


In [ ]:
cache_report = benchmark(
    engine, {"state": MESSAGE, "questions": questions}, images=[], iterations=3
)
print("Cached:", round(cache_report["cached_median_seconds"] * 1000, 1), "ms")
print("Uncached:", round(cache_report["uncached_median_seconds"] * 1000, 1), "ms")
print("Observed speedup:", round(cache_report["observed_speedup"], 2), "x")
print("Maximum support difference:", cache_report["max_candidate_support_difference"])
print("Same choices:", cache_report["same_choices"])
print("Tokens reused:", cache_report["cached_usage"]["reused_input_tokens"])

reordered = dict(reversed(list(questions.items())))
reordered["department"] = Choice(
    questions["department"].instructions,
    dict(reversed(list(questions["department"].criteria.items()))),
)
order_check = compare_results(text_result, engine.decide(MESSAGE, reordered))
print("Candidate order check:", order_check)


## 5. Save results and release the model

The next cell saves `openjev-bonsai-results.json`. Download it from Colab's Files panel to keep it. Rerun the model-loading cell before making more decisions after cleanup.


In [ ]:
saved_results = {
    "model_revision": MODEL_REVISION,
    "tokenizer_revision": TOKENIZER_REVISION,
    "runtime": "prism-b10709-9a9394a",
    "gguf_file": GGUF_FILE,
    "text": text_result.to_dict(),
    "benchmark": cache_report,
    "candidate_order": order_check,
}
output_path = Path("openjev-bonsai-results.json")
output_path.write_text(json.dumps(saved_results, indent=2, allow_nan=False))
print("Saved:", output_path.resolve())
engine.close()
del engine
gc.collect()


Scores are normalized independent candidate support, not calibrated probabilities of correctness. This example is a runtime and caching check, not a model-quality benchmark. Bonsai's published thinking-mode benchmarks do not measure this direct token-scoring setup.

[OpenJev README](https://github.com/mjdileep/OpenJev) · [Validation results](https://github.com/mjdileep/OpenJev/blob/main/docs/validation.md) · [Prism runtime source](https://github.com/PrismML-Eng/llama.cpp/tree/9a9394a895b96003ca842a6041cb28ac49a108f7)
